# Original PatchTST 
### imported from transformers

## Import module

In [21]:
import os, math, gc, warnings
# GPU control
os.environ["CUDA_VISIBLE_DEVICES"] = "2"
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import PatchTSTConfig, PatchTSTForPrediction

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [22]:
PATH_TRAIN = "./data_filtering/filtered/train.csv"           
PATH_TEST00 = "./data_filtering/filtered/TEST_00.csv"  
SAVE_DIR = "./Model/"    # 모델 저장 폴더

# 대회 규격
CONTEXT_LEN = 28
PRED_LEN = 7
CHANNELS = 1    # univarate : sales

# 학습 설정
'''
BATCH_SIZE = 256
EPOCHS = 20         # 처음엔 짧게
LR = 1e-3
WEIGHT_DECAY = 1e-2
GRAD_CLIP = 1.0
'''
EPOCHS = 150
BATCH_SIZE = 512          # 메모리 부족하면 256 유지 + 아래 ACCUM 사용
LR = 3e-4                 # 용량 키우면 초기 lr 조금 낮추는 게 안전
WEIGHT_DECAY = 5e-2       # 0.05
GRAD_CLIP = 0.5

VAL_DAYS = CONTEXT_LEN + PRED_LEN  # 최소 검증 블록(28+7=35일)
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

## Load dataset & sort

In [23]:
train_df = pd.read_csv(PATH_TRAIN)
train_df["date"] = pd.to_datetime(train_df["date"])
train_df.loc[train_df["sales"] < 0, "sales"] = 0

# 파일에 따라 컬럼명이 store_menu 또는 store_menu_id일 수 있음. 없으면 만들어줌.
if "store_menu" not in train_df.columns:
    if "store_menu_id" in train_df.columns:
        train_df["store_menu"] = train_df["store_menu_id"]
    else:
        train_df["store_menu"] = train_df["store"] + "_" + train_df["menu"]

# 정렬
train_df = train_df.sort_values(["store_menu", "date"]).reset_index(drop=True)

# 각 시리즈 길이 확인
lens = train_df.groupby("store_menu")["date"].count().sort_values()

In [24]:
scalers = {}  # sid -> (mu, sigma)

def fit_scaler_partial(y: np.ndarray, val_days: int):
    cut = max(0, len(y) - val_days)
    base = y[:cut] if cut > 0 else y
    mu = float(base.mean()) if len(base) else 0.0
    sigma = float(base.std()) if len(base) else 1.0
    if sigma < 1e-8: sigma = 1.0
    return mu, sigma

for sid, g in train_df.groupby("store_menu"):
    y = g["sales"].values.astype("float32")
    mu, sigma = fit_scaler_partial(y, VAL_DAYS)
    scalers[sid] = (mu, sigma)

len(scalers), list(scalers.items())[:1]


(193, [('느티나무 셀프BBQ_1인 수저세트', (5.1911468505859375, 7.603192329406738))])

## Window dataset

In [25]:
class TSWindows(Dataset):
    def __init__(self, df, scalers, context_len=28, pred_len=7, split="train"):
        self.X, self.Y = [], []
        for sid, g in df.groupby("store_menu"):
            y = g["sales"].values.astype("float32")
            mu, sigma = scalers.get(sid, (0.0, 1.0))
            y_scaled = (y - mu) / sigma

            # 검증 구간 시작점
            cut = max(0, len(y) - (context_len + pred_len))
            if split == "train":
                end = max(context_len, cut)  # target이 검증 구간에 걸리지 않도록
                # train 윈도우 생성 범위: t in [context_len, end - pred_len]
                for t in range(context_len, end):
                    if t + pred_len <= end:
                        x = y_scaled[t-context_len:t]
                        fut = y_scaled[t:t+pred_len]
                        self.X.append(x[:, None])     # (28,1)
                        self.Y.append(fut[:, None])   # (7,1)
            else:
                # val 윈도우: 검증 블록 전부 사용(보통 1개 이상)
                t0 = max(context_len, cut)
                for t in range(t0, len(y) - pred_len + 1):
                    x = y_scaled[t-context_len:t]
                    fut = y_scaled[t:t+pred_len]
                    self.X.append(x[:, None])
                    self.Y.append(fut[:, None])

        self.X = np.asarray(self.X, dtype="float32")
        self.Y = np.asarray(self.Y, dtype="float32")

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return torch.from_numpy(self.X[idx]), torch.from_numpy(self.Y[idx])

train_ds = TSWindows(train_df, scalers, CONTEXT_LEN, PRED_LEN, split="train")
val_ds   = TSWindows(train_df, scalers, CONTEXT_LEN, PRED_LEN, split="val")

len(train_ds), len(val_ds)


(89359, 5597)

In [26]:
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

## Loading pretrained patchTST

In [27]:
# === 6) Pretrained PatchTST 로드(백본 이식) + 풀 파인튜닝 — FIXED ===
from packaging import version
import torch
from transformers import PatchTSTConfig, PatchTSTForPrediction

BASE_REPO = "ibm-research/patchtst-etth1-pretrain"  # 마스킹 프리트레인 백본

cfg = PatchTSTConfig.from_pretrained(BASE_REPO)
cfg.context_length = CONTEXT_LEN
cfg.prediction_length = PRED_LEN
cfg.num_input_channels = CHANNELS

# after: cfg.context_length, cfg.prediction_length, cfg.num_input_channels
def _set(cfg, k, v):
    if hasattr(cfg, k):
        setattr(cfg, k, v); print(f"cfg.{k} -> {getattr(cfg,k)}")

_set(cfg, "hidden_size", 512)          # 256 → 512
_set(cfg, "num_hidden_layers", 8)      # 4~6 → 8
_set(cfg, "num_attention_heads", 8)    # 8 유지 or 16(메모리 여유 많으면)
_set(cfg, "patch_len", 14)             # 7/14가 주간성에 잘 맞음
_set(cfg, "stride", 1)                 # 1이 보통 성능↑, 메모리만 버티면 고정
_set(cfg, "dropout", 0.2)              # 0.2~0.3


def load_patchtst(repo: str, cfg: PatchTSTConfig):
    # 1) safetensors 먼저
    try:
        print("Trying safetensors weights...")
        m = PatchTSTForPrediction.from_pretrained(
            repo,
            config=cfg,
            ignore_mismatched_sizes=True,
            use_safetensors=True,   # 핵심
        )
        print("Loaded safetensors weights.")
        return m
    except Exception as e_s:
        print("Safetensors load failed:", repr(e_s))

    # 2) .bin 로드는 torch>=2.6 필요
    if version.parse(torch.__version__) >= version.parse("2.6.0"):
        print(f"torch {torch.__version__} detected (>=2.6). Trying .bin weights...")
        return PatchTSTForPrediction.from_pretrained(
            repo,
            config=cfg,
            ignore_mismatched_sizes=True,
            use_safetensors=False,
        )
    else:
        raise RuntimeError(
            f"이 체크포인트 저장소에 safetensors가 없고, torch=={torch.__version__} < 2.6 입니다.\n"
            "해결책:\n"
            "  (A) torch를 2.6 이상으로 업그레이드하거나,\n"
            "  (B) safetensors 가중치가 있는 다른 PatchTST 체크포인트를 사용하세요.\n"
            "예: ALT_REPO = 'namctin/patchtst_etth1_forecast'  # HF에 safetensors 있음(대개)\n"
            "     PatchTSTForPrediction.from_pretrained(ALT_REPO, config=cfg, ignore_mismatched_sizes=True, use_safetensors=True)\n"
        )

model = load_patchtst(BASE_REPO, cfg).to(device)

# 옵티마이저/스케줄러 (기존 그대로)
'''
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
'''
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.98))
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=30, T_mult=2)
scaler_amp = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

criterion = torch.nn.L1Loss()  # MAE


cfg.hidden_size -> 512
cfg.num_hidden_layers -> 8
cfg.num_attention_heads -> 8
cfg.stride -> 1
cfg.dropout -> 0.2
Trying safetensors weights...


Some weights of PatchTSTForPrediction were not initialized from the model checkpoint at ibm-research/patchtst-etth1-pretrain and are newly initialized: ['head.projection.bias', 'head.projection.weight', 'model.encoder.embedder.input_embedding.bias', 'model.encoder.embedder.input_embedding.weight', 'model.encoder.layers.0.ff.0.bias', 'model.encoder.layers.0.ff.0.weight', 'model.encoder.layers.0.ff.3.bias', 'model.encoder.layers.0.ff.3.weight', 'model.encoder.layers.0.norm_sublayer1.batchnorm.bias', 'model.encoder.layers.0.norm_sublayer1.batchnorm.num_batches_tracked', 'model.encoder.layers.0.norm_sublayer1.batchnorm.running_mean', 'model.encoder.layers.0.norm_sublayer1.batchnorm.running_var', 'model.encoder.layers.0.norm_sublayer1.batchnorm.weight', 'model.encoder.layers.0.norm_sublayer3.batchnorm.bias', 'model.encoder.layers.0.norm_sublayer3.batchnorm.num_batches_tracked', 'model.encoder.layers.0.norm_sublayer3.batchnorm.running_mean', 'model.encoder.layers.0.norm_sublayer3.batchnorm.r

Loaded safetensors weights.


## Training

In [28]:
# === Training (gradient accumulation + masked loss + early stop + 안전한 스텝 처리) ===
import os, torch
from torch import nn

# 하이퍼 파라미터(필요시 네 값으로 덮어써)
ACCUM_STEPS = 2          # 유효 배치 = BATCH_SIZE * ACCUM_STEPS
GRAD_CLIP    = GRAD_CLIP # 기존 값 사용
EPOCHS       = EPOCHS    # 기존 값 사용
SAVE_DIR     = SAVE_DIR  # 기존 값 사용

# 손실: 0 매출 마스크 + SmoothL1 혼합
sl1 = nn.SmoothL1Loss(beta=1.0)
def masked_mae(pred, y, eps=1e-6):
    m = (y != 0).float()
    return (torch.abs(pred - y) * m).sum() / (m.sum() + eps)

# 검증 루프(손실 정의와 동일하게)
@torch.no_grad()
def evaluate(loader):
    model.eval()
    loss_sum = 0.0
    n = 0
    for xb, yb in loader:
        xb = xb.to(device)   # (B, 28, 1)
        yb = yb.to(device)   # (B, 7, 1)
        out = model(past_values=xb)
        pred = out.prediction_outputs
        loss = 0.7 * masked_mae(pred, yb) + 0.3 * sl1(pred, yb)
        loss_sum += loss.item() * xb.size(0)
        n += xb.size(0)
    return loss_sum / max(1, n)

# 입력 노이즈로 일반화 살짝(+)
ADD_INPUT_NOISE = True
NOISE_SCALE = 0.01  # 0.005~0.01 사이 권장

best_val = float("inf")
bad, PATIENCE = 0, 20

for epoch in range(1, EPOCHS + 1):
    model.train()
    opt.zero_grad(set_to_none=True)
    step_in_epoch = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        # 약간의 입력 노이즈(훈련시에만)
        if ADD_INPUT_NOISE:
            std_per_seq = xb.std(dim=1, keepdim=True)  # (B,1,1)
            xb = xb + NOISE_SCALE * std_per_seq * torch.randn_like(xb)

        with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
            out = model(past_values=xb)
            pred = out.prediction_outputs
            loss = 0.7 * masked_mae(pred, yb) + 0.3 * sl1(pred, yb)

        # gradient accumulation
        scaler_amp.scale(loss / ACCUM_STEPS).backward()
        step_in_epoch += 1

        if step_in_epoch % ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler_amp.step(opt)
            scaler_amp.update()
            opt.zero_grad(set_to_none=True)

        # OneCycleLR을 쓰는 경우엔 여기서 scheduler.step()
        # if isinstance(scheduler, torch.optim.lr_scheduler.OneCycleLR):
        #     scheduler.step()

    # 에폭이 ACCUM_STEPS의 배수가 아니면 남은 그라드를 마지막에 처리
    if step_in_epoch % ACCUM_STEPS != 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler_amp.step(opt)
        scaler_amp.update()
        opt.zero_grad(set_to_none=True)

    # CosineAnnealingWarmRestarts / CosineAnnealingLR 등은 에폭 말에 한 번
    if not isinstance(scheduler, torch.optim.lr_scheduler.OneCycleLR):
        scheduler.step()

    # 검증
    val_mae = evaluate(val_loader)

    # 베스트 저장
    if val_mae < best_val:
        best_val = val_mae
        bad = 0
        os.makedirs(SAVE_DIR, exist_ok=True)
        model.save_pretrained(SAVE_DIR)
    else:
        bad += 1

    print(f"[{epoch:03d}/{EPOCHS}] val(MAE-like)={val_mae:.5f} | best={best_val:.5f} | bad={bad}/{PATIENCE}")

    # 얼리스톱
    if bad >= PATIENCE:
        print(f"Early stop at epoch {epoch}")
        break

best_val


[001/150] val(MAE-like)=0.50665 | best=0.50665 | bad=0/20
[002/150] val(MAE-like)=0.47390 | best=0.47390 | bad=0/20
[003/150] val(MAE-like)=0.44605 | best=0.44605 | bad=0/20
[004/150] val(MAE-like)=0.44631 | best=0.44605 | bad=1/20
[005/150] val(MAE-like)=0.44278 | best=0.44278 | bad=0/20
[006/150] val(MAE-like)=0.44425 | best=0.44278 | bad=1/20
[007/150] val(MAE-like)=0.43965 | best=0.43965 | bad=0/20
[008/150] val(MAE-like)=0.43749 | best=0.43749 | bad=0/20
[009/150] val(MAE-like)=0.43799 | best=0.43749 | bad=1/20
[010/150] val(MAE-like)=0.43830 | best=0.43749 | bad=2/20
[011/150] val(MAE-like)=0.43825 | best=0.43749 | bad=3/20
[012/150] val(MAE-like)=0.43699 | best=0.43699 | bad=0/20
[013/150] val(MAE-like)=0.43731 | best=0.43699 | bad=1/20
[014/150] val(MAE-like)=0.43330 | best=0.43330 | bad=0/20
[015/150] val(MAE-like)=0.43288 | best=0.43288 | bad=0/20
[016/150] val(MAE-like)=0.43144 | best=0.43144 | bad=0/20
[017/150] val(MAE-like)=0.43332 | best=0.43144 | bad=1/20
[018/150] val(

0.4183177861578234

## Inference

In [29]:
# 가장 좋은 체크포인트 로드
model = PatchTSTForPrediction.from_pretrained(SAVE_DIR).to(device).eval()

In [30]:
import os
import pandas as pd
import numpy as np
import torch
import pickle

# 전제: 다음 변수/객체가 이전 셀에서 이미 존재해야 함
# - model : PatchTSTForPrediction (eval 모드)
# - scalers : dict[sid] = (mu, sigma)
# - CONTEXT_LEN = 28, PRED_LEN = 7
# - device

def _ensure_store_menu(df: pd.DataFrame) -> pd.DataFrame:
    if "store_menu" not in df.columns:
        if "store_menu_id" in df.columns:
            df["store_menu"] = df["store_menu_id"]
        else:
            df["store_menu"] = df["store"].astype(str) + "_" + df["menu"].astype(str)
    return df

def predict_for_test_df(test_df: pd.DataFrame,
                        model,
                        scalers: dict,
                        context_len: int = 28,
                        pred_len: int = 7) -> pd.DataFrame:
    test_df = test_df.copy()
    test_df["date"] = pd.to_datetime(test_df["date"])
    test_df = _ensure_store_menu(test_df)
    test_df = test_df.sort_values(["store_menu", "date"])
    
    # 예측할 날짜 인덱스(마지막 날짜 다음날부터 7일)
    last_date = test_df["date"].max()
    future_index = pd.date_range(last_date + pd.Timedelta(days=1), periods=pred_len, freq="D")

    # store_menu 별로 (28 -> 7) 예측
    sids = sorted(test_df["store_menu"].unique())
    pred_mat = np.zeros((pred_len, len(sids)), dtype="float32")

    model.eval()
    with torch.no_grad():
        for j, sid in enumerate(sids):
            g = test_df[test_df["store_menu"] == sid]
            y_last28 = g["sales"].values.astype("float32")
            if len(y_last28) != context_len:
                raise ValueError(f"{sid}: 입력 길이 {len(y_last28)} != {context_len}")

            mu, sigma = scalers.get(sid, (0.0, 1.0))
            x = ((y_last28 - mu) / sigma).astype("float32")
            x = torch.from_numpy(x).to(device).unsqueeze(0).unsqueeze(-1)  # (1, 28, 1)

            out = model(past_values=x)
            yhat = out.prediction_outputs.squeeze(0).squeeze(-1).detach().cpu().numpy()  # (7,)
            yhat = yhat * sigma + mu
            yhat = np.clip(yhat, 0, None)  # 음수 방지
            pred_mat[:, j] = yhat

    pred_df = pd.DataFrame(pred_mat, index=future_index, columns=sids)
    return pred_df

# ===== 실제 실행: TEST_00 ~ TEST_09 일괄 처리 =====
test_dir = "./data_filtering/filtered/"  # 환경에 맞게 조정
preds_by_test = {}      # {"TEST_00": DataFrame(...), ...}

for i in range(10):
    fname = f"TEST_{i:02d}.csv"
    fpath = os.path.join(test_dir, fname)
    if not os.path.exists(fpath):
        print(f"[SKIP] {fname} 없음")
        continue
    print(f"[RUN ] {fname} 예측 중...")
    df_test = pd.read_csv(fpath)
    pred_df = predict_for_test_df(df_test, model, scalers, CONTEXT_LEN, PRED_LEN)
    preds_by_test[f"TEST_{i:02d}"] = pred_df


print(f"완료: 총 {len(preds_by_test)}개 TEST 파일 예측 저장")


[RUN ] TEST_00.csv 예측 중...
[RUN ] TEST_01.csv 예측 중...
[RUN ] TEST_02.csv 예측 중...
[RUN ] TEST_03.csv 예측 중...
[RUN ] TEST_04.csv 예측 중...
[RUN ] TEST_05.csv 예측 중...
[RUN ] TEST_06.csv 예측 중...
[RUN ] TEST_07.csv 예측 중...
[RUN ] TEST_08.csv 예측 중...
[RUN ] TEST_09.csv 예측 중...
완료: 총 10개 TEST 파일 예측 저장


In [31]:
# 1) 0 < 값 ≤ 1 인 셀은 1로 치환
preds_by_test_adj = {}
for test_key, df_pred in preds_by_test.items():
    df_adj = df_pred.copy()
    mask = (df_adj >= 0) & (df_adj <= 1)
    df_adj[mask] = 1.0
    preds_by_test_adj[test_key] = df_adj

# 2) sample_submission_date.csv 로드
sample_path = "./result/sample_submission_date.csv"
sub_df = pd.read_csv(sample_path, parse_dates=["date"])

# 컬럼명이 store_menu 형식이라고 가정
store_menu_cols = [c for c in sub_df.columns if c != "date"]

# 3) preds_by_test_adj에서 값 채우기
# test_key -> 날짜 매핑: TEST_00, TEST_01, ...
for test_key, df_pred in preds_by_test_adj.items():
    # df_pred index: 예측 날짜
    for pred_date in df_pred.index:
        # sample_submission_date.csv 에서 동일 date 찾기
        if pred_date not in sub_df["date"].values:
            continue
        row_mask = sub_df["date"] == pred_date
        # 각 store_menu에 맞춰 값 대입
        for col in df_pred.columns:
            if col in sub_df.columns:
                sub_df.loc[row_mask, col] = df_pred.loc[pred_date, col]

# 4) 저장
out_path = "./result/submission_filled.csv"
os.makedirs(os.path.dirname(out_path), exist_ok=True)
sub_df.to_csv(out_path, index=False, encoding='utf-8-sig')
print(f"저장 완료: {out_path}")


저장 완료: ./result/submission_filled.csv
